# Reachability-based BLAs for intersection crossings

An executable illustration of the planned LTMS example. The manager offers
space–time reservations; a vehicle's dynamics and crossing task qualify the
offers; finite bargaining selects one exact owner promise and price. Earlier
BLAs survive every subsequent attempt.

**Prototype for review.** This notebook uses the extracted `ltms_reachability`
package, `protocol` contract algebra and the actual bargaining implementation
from `protocol.ipynb`. It demonstrates numerical corridors and a restricted,
deterministic client controller. It establishes no disturbance-robust safety
filter or general continuous-time HJ error certificate.


In [1]:
from pathlib import Path
from IPython.display import Markdown, display
import json
import os
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src' / 'ltms_reachability').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from ltms_bla.scenario import run_scenario
from ltms_bla.records import CrossingService
from ltms_bla.presentation import results_table, plot_scenario, export_report
from protocol.presentation import display_timeline

OUTPUT = Path(os.environ.get('LTMS_OUTPUT', ROOT / 'artifacts' / 'ltms_blas'))
OUTPUT.mkdir(parents=True, exist_ok=True)


## 1. Plant, service and client

Use the copied four-way LTMS map and Bicycle5D state $z=(x,y,h,\delta,v)$:

$$\dot x=v\cos h,\quad \dot y=v\sin h,\quad
\dot h=v\tan\delta/0.32,\quad \dot\delta=u_1,\quad \dot v=u_2.$$

The numerical chart has $h\in[-0.5,0.5]$, $\delta\in[-0.15,0.15]$,
$v\in[0.3,0.6]$, $|u_1|\le0.1$, $|u_2|\le0.15$. Westbound motion rotates
this chart by $\pi$. This is a route-local five-state computation, without a
periodic heading seam. It is not a reproduction of the thesis's global grid.

Each service fixes a route, allowed departure slots, deadline and exact entry
state. In local coordinates the nominal controller starts at
$(-1.6,-0.4,0,0,0.45)$ and applies $u=(0,0)$ for 7 s. Its analytic trajectory
is $(-1.6+0.45\tau,-0.4,0,0,0.45)$ and clears the managed square
$[-1.2,1.2]^2$ before departure + 7. A 0.32 m by 0.16 m toy body fits in a
declared 0.18 m disk. These are illustrative dimensions, not a measured vehicle.

The fixed client model selects one covered reference from its declared finite
slot library and follows this controller under perfect initial state and
dynamics. Thus all alternative offers for a selected service use the **same**
client model. No turns or uncertain tracking are claimed.


## 2. Reachability → corridor → physical evidence

`LTMSPlanner` runs the four LTMS passes and a native pyspect
`TLT(UNTIL(road, clear_in_[7,8]))`, realized by `TVHJImpl`. The target is time
masked; the native backend also applies the constraint at the witness.
The full array axes are $(t,x,y,h,\delta,v)$.

The offered corridor is the final tube restricted to the straight lane strip
$|y+0.4|\le0.06$. A single nominal execution is checked inside this conjunction
at 0.025 s resolution. The cropped tube is not asserted to be a viability
kernel for arbitrary controls. Full tubes remain in immutable hashed local
artifacts; native TLT and corridor values are checked at the specified full
entry state, rather than only at its xy projection.

Workspace reservations retain time and inflate the corridor projection by the
body, a 0.01 m extra allowance, grid-cell bounds and componentwise motion over
each time slab. Checks reject contact of closed raster cells, verify complete
swept reference-body boxes, and minimize pairwise nominal body separation
analytically between samples. Extra allocation margin does not prove a tracking
controller. An empty numerical result remains unresolved unless an independent
necessary bound proves impossibility.

The coordinator runs on Python 3.14. The persistent numerical worker uses
Python 3.13 with `pyspect[hj_reachability]==1.2` on this Intel Mac. The first
execution computes the actual HJ artifacts; later runs verify and reuse them.
The default worker command uses `uv` and includes Pillow and Matplotlib;
`LTMS_WORKER_PYTHON` can select an
already provisioned interpreter. Worker progress is written to `OUTPUT/worker.log`.


In [2]:
report = run_scenario(ROOT, OUTPUT)
meta = report['final']['metadata']
display(Markdown(
    f"**Backend:** {meta['backend']}  \n"
    f"**Grid:** {meta['shape']}; one tube {meta['tube_mib']:.1f} MiB  \n"
    f"**Cold numerical preparation:** {meta['cold_seconds']:.1f} s; "
    f"cache used this run: {meta['cache_hit']}  \n"
    f"**Peak worker memory during preparation:** {meta['peak_worker_mib']:.0f} MiB  \n"
    f"**Versions:** `{meta['config']['versions']}`  \n"
    f"**Entry values:** native TLT {meta['checks']['tlt_entry_value']:.5f}, "
    f"offered corridor {meta['checks']['corridor_entry_value']:.5f}"
))


**Backend:** pyspect TVHJImpl / Bicycle5D  
**Grid:** [37, 31, 61, 9, 5, 7]; one tube 84.1 MiB  
**Cold numerical preparation:** 177.5 s; cache used this run: False  
**Peak worker memory during preparation:** 1359 MiB  
**Versions:** `{'pyspect': '1.2', 'hj-reachability': '0.7.0', 'jax': '0.4.38', 'numpy': '2.5.4'}`  
**Entry values:** native TLT -0.11535, offered corridor -0.06000

## 3. One selected service, several owner offers

Let $\xi_s$ be a service's declared reference trajectory for slot $s$ and let
$r_j(t)$ be the manager's spatial allocation. Define whole-trajectory predicates

$$a_s := \forall t\in[s,s+7]:\ \mathrm{Body}(\xi_s(t))\subseteq r_j(t),
\qquad S_a(\theta)=\bigvee_{s\in D_\theta:\,s+7\le d_\theta} a_s.$$

Let $B_j$ mean the client body remains in its allocation while present, and
$S_u(\theta)$ mean road-compliant clearance by its deadline. The client and task
are $C_{client}=\langle S_a\mid B_j\land S_u\rangle$ and
$R_{task}=\langle\top\mid B_j\land S_u\rangle$. The library's quotient leaves
the owner guarantee $S_a$; conjunction with standing commitments gives
$\langle\top\mid L\land S_a\rangle$.

For a qualified corridor, $H_\kappa$ means the manager covers its full inflated
workspace reservation. The offer stores $K_\kappa=\langle\top\mid
H_\kappa\land a_s\rangle$. The swept-body check establishes that the reference
reservation is contained in the promised corridor reservation, so $a_s$ is
an explicit consequence of $H_\kappa$. Hence the offer implies the fixed
$S_a$; disjoint owner reservations establish compatibility separately.

The code uses Boolean atoms for these whole-trajectory predicates, with these
geometric meanings. It does **not** apply the charging booking SAT procedure to
spatial contracts, or implement a general spatial temporal-logic decision
procedure. `Contract.guarantee` stores the saturated guarantee internally.


In [3]:
service = CrossingService('B', 'west', (0, 1), 9)
client, task, quotient = service.contracts()
display(Markdown('For B, both slots belong to one fixed selected service:'))
display(client, task, quotient)
display(Markdown(results_table(report)))


For B, both slots belong to one fixed selected service:

Contract(assumption=Or(operands=(Proposition(name='covers_ref_eabbb5fd2ec118b1_0'), Proposition(name='covers_ref_eabbb5fd2ec118b1_1'))), guarantee=Or(operands=(And(operands=(Proposition(name='access_rule_B'), Proposition(name='road_and_clear_by_9_eabbb5fd2ec118b1'))), And(operands=(Not(operand=Proposition(name='covers_ref_eabbb5fd2ec118b1_0')), Not(operand=Proposition(name='covers_ref_eabbb5fd2ec118b1_1')))))))

Contract(assumption=Boolean(value=True), guarantee=And(operands=(Proposition(name='access_rule_B'), Proposition(name='road_and_clear_by_9_eabbb5fd2ec118b1'))))

Contract(assumption=And(operands=(Or(operands=(Proposition(name='access_rule_B'), Not(operand=Proposition(name='covers_ref_eabbb5fd2ec118b1_0')))), Or(operands=(Proposition(name='access_rule_B'), Not(operand=Proposition(name='covers_ref_eabbb5fd2ec118b1_1')))), Or(operands=(Proposition(name='road_and_clear_by_9_eabbb5fd2ec118b1'), Not(operand=Proposition(name='covers_ref_eabbb5fd2ec118b1_0')))), Or(operands=(Proposition(name='road_and_clear_by_9_eabbb5fd2ec118b1'), Not(operand=Proposition(name='covers_ref_eabbb5fd2ec118b1_1')))))), guarantee=Or(operands=(Proposition(name='covers_ref_eabbb5fd2ec118b1_0'), Proposition(name='covers_ref_eabbb5fd2ec118b1_1'))))

| Client | Outcome | Qualified offers | Selected departure | Price | Prior BLAs retained |
|:--|:--|--:|--:|--:|:--|
| A | COMMITTED | 1 | 0 | 15/2 | True |
| B | COMMITTED | 2 | 0 | 15/2 | True |
| C | NO-COMPATIBLE-OFFER | 0 | — | — | True |
| D | PRICE-NO-BLA | 2 | — | — | True |
| E | COMMITTED | 1 | 20 | 15/2 | True |

## 4. Serial attempts and bargaining

A first crossing is accepted. B has two compatible corridors with overlapping
physical times and the opposite direction. C conflicts with A's retained
reservation. D has compatible corridors but its price negotiation fails.
E's original deadline is independently impossible: its minimum distance to the
goal is 3 m, while 0.6 m/s permits only 1.2 m in 2 s. Its explicitly permitted
later service is admitted afresh, with a new fixed client template.

Bargaining executes the **same function body** as `protocol.ipynb`, replacing
only the charging-specific offer eligibility validator. Utilities are
$q-c_\kappa$ and $v_\kappa-q$. For the two-slot example the early/later offers
have illustrative costs 5/2 and values 10/8; D instead has values 1/0. These
declared preferences create a price/delay choice; they are not fitted traffic
economics. Initial targets, minima, bounded concessions and the final response
rule are retained. No HJ solve runs per bargaining round.

The finite family reuses one verified base tube under time translation and map
rotation. Whole reservations are filtered against each frozen accepted snapshot.
Conflicting templates are rejected, rather than recomputing more permissive
corridors around existing reservations.


In [4]:
round_rows = ['| Client | Round | Proposer | Price | Owner utility | Client utility | Accepted |',
              '|:--|--:|:--|--:|--:|--:|:--|']
for attempt in report['attempts']:
    for r in attempt['rounds']:
        round_rows.append(f"| {attempt['client']} | {r['number']} | {r['proposer']} | "
                          f"{r['price']} | {r['utilities']['owner']} | {r['utilities']['client']} | {r['accepted']} |")
display(Markdown('\n'.join(round_rows)))


| Client | Round | Proposer | Price | Owner utility | Client utility | Accepted |
|:--|--:|:--|--:|--:|--:|:--|
| A | 1 | owner | 9 | 4 | 1 | False |
| A | 2 | client | 15/2 | 5/2 | 5/2 | True |
| B | 1 | owner | 9 | 4 | 1 | False |
| B | 2 | client | 15/2 | 5/2 | 5/2 | True |
| D | 1 | owner | 9 | 4 | -8 | False |
| D | 2 | client | 1/2 | -9/2 | 1/2 | False |
| D | 3 | owner | 15/2 | 5/2 | -13/2 | False |
| E | 1 | owner | 9 | 4 | 1 | False |
| E | 2 | client | 15/2 | 5/2 | 5/2 | True |

In [5]:
figure = plot_scenario(report)
display_timeline(figure)
export_report(report, figure, OUTPUT)


LTMS reachability-based BLAs

## 5. What was checked

Acceptance stores the exact negotiated offer object and price, without
regeneration. Artifact hashes bind the full five-state corridor, reservation
and model; stale snapshots and substituted records are rejected. Every attempt
checks retention of every earlier BLA and artifact, including price failure.
Reservations remain allocated to lease end even after nominal exit.

The figure's shading is the time union of the **actual** spatial reservations,
shown only for visualization. Compatibility uses their full time-dependent
volumes. The right-hand plot fixes heading, steering and speed; it is not a
projection over unspecified entry states.


In [6]:
assert [a['status'] for a in report['attempts']] == [
    'COMMITTED', 'COMMITTED', 'NO-COMPATIBLE-OFFER', 'PRICE-NO-BLA', 'COMMITTED']
assert all(a['previous_preserved'] for a in report['attempts'])
assert all(a['accepted']['exact_object'] for a in report['attempts'] if a['accepted'])
assert len(report['attempts'][1]['offers']) == 2
assert report['attempts'][-1]['services'][0]['candidates'][0]['status'] == 'PROVED_IMPOSSIBLE'
assert all(p['minimum_body_gap'] is None or p['minimum_body_gap'] > 0
           for p in report['final']['pair_checks'])
assert report['refinement']['entry_membership_stable']
display(Markdown(f"All lifecycle checks pass. Minimum simultaneous body clearance: "
                 f"{min(p['minimum_body_gap'] for p in report['final']['pair_checks'] if p['minimum_body_gap'] is not None):.2f} m."))
display(Markdown('Temporal refinement check: `' + json.dumps(report['refinement']) + '`.'))


All lifecycle checks pass. Minimum simultaneous body clearance: 0.44 m.

Temporal refinement check: `{"base_digest": "aa88e5747a2e779865587b65afaad254e3c3d9bc1af3b0fea37de64b294f7107", "coarse_dt": 0.25, "fine_dt": 0.125, "coarse_value": -0.11534937471151352, "fine_value": -0.11458324640989304, "entry_membership_stable": true, "seconds": 53.28650685798493, "cache_hit": true, "scope": "temporal refinement only; not a numerical error bound"}`.

## Scope and reproduction

This is a finite straight-crossing example under perfect initial state and
dynamics. The conditional client interpretation has a concrete nominal
controller; numerical HJ membership and the sampled corridor construction do
not establish general closed-loop invariance, uncertain tracking or complete
coverage of feasible offers. The refinement check halves the temporal grid;
it is a sensitivity check, not an error bound. The route-local map computation
does not complete the planned full-resolution legacy LTMS reproduction.

From the repository root:

```sh
uv run --group notebooks python scripts/run_ltms_notebook.py --save
# Force an independent cold numerical run into a new directory:
uv run --group notebooks python scripts/run_ltms_notebook.py --output /tmp/ltms-fresh
```

Outputs include `report.json`, exact offer records, hashed numerical arrays,
`results.md`, an interactive HTML figure and standalone SVG/PNG figures.
They are reproducible local artifacts, excluded from Git. The notebook retains
compact, verified results and the native interactive figure.
